# Agent Architecture: Reflection + Light Tool Use

This notebook uses the shared versioned parquet dataset instead of generating a fresh example in memory.

That keeps the walkthrough reproducible and aligned with evaluation.

## Pipeline shape

The project implements a compact reflection pipeline:

1. Analyzer planning step chooses which tools to call.
2. Tool layer runs lightweight statistical checks.
3. Analyzer writes the first diagnosis.
4. Critic reviews the diagnosis.
5. Reviser writes the final answer.

All stages return structured outputs, and LangSmith can trace every step.

## Data contract used by the agent

The stored experiment summaries are richer than before. In addition to the original pre/post summary fields, each record now includes:

- campaign-level metrics such as impressions, clicks, spend, conversions, and revenue
- arm-level metrics such as control and treatment impressions, clicks, spend, revenue, users, and conversions
- latent factors that describe hidden data-generation conditions
- hidden `true_*` values reserved for evaluation only

The important design choice is that the agent still reasons over the observed summary fields, while the `true_*` fields remain out-of-band evaluation metadata.

In [ ]:
from rct_diagnosis_agent.agent import build_default_agent
from rct_diagnosis_agent.data import DatasetConfig, load_or_generate_data

experiments = load_or_generate_data(DatasetConfig(count=25, seed=7))
experiment = experiments[0]
agent = build_default_agent()

## Inspect the stored experiment

This is the exact observed summary object that will be passed into the agent prompt. Hidden evaluation-only fields are automatically removed by `to_prompt_dict()`.

In [ ]:
experiment.to_prompt_dict()

## Run the pipeline

If LangSmith tracing is enabled, the run will show the plan, tool calls, analyzer output, critique, and revision as separate spans.

In [ ]:
result = agent.run(experiment)
result.model_dump()

## Why the shared dataset matters

Because the walkthrough reads from the same stored dataset as the evaluation notebook, you can trace a single experiment here and then compare that same example's behavior in the benchmark later.

That makes the project easier to inspect and less sensitive to accidental reruns.